#  Pure data case

We have three hyperparameters for the model:

- Relationship between $\sigma_s$ and $\sigma_0$
  - These follow the relation $\sigma_x^2 = K\sigma_0^2\sigma_s^2$, where $K$ is the number of atoms (or rows in the components matrix)
  - We can set one of them to $c\sigma_{\epsilon}$, where $c$ is some constant deciding how you want to tune reconstruction penalty v/s regularisation
  - Or we can make them equal, and set both as: $\sigma_0 = \sigma_s = \frac{\sigma_x}{\sqrt{K}}$
  - This removes $\sigma_{\epsilon}$ from the equation, do note that $\sigma_{\epsilon}$ is itself set with respect to $\sigma_x$, which we assume has 1 std.
  - If you increase $K$, $\sigma_0$ decreases. This can make it come closer to $\sigma_{\epsilon}$ which is not tuned wrt $K$

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import *
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
)
from pt_to_api import disjoint_ae, disjoint_ae_learned_sig
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import numpy as np
from torch import nn
from torch import optim
import warnings
from dataclasses import dataclass
from typing import Any
import math
import gc
import pandas as pd


MODE = "light"

In [ ]:
class MeanPerDimGlobalStdScaler:
    def __init__(self):
        self.means_ = None
        self.global_std_ = None

    def fit(self, X):
        X = np.array(X)
        self.means_ = X.mean(axis=0)
        self.global_std_ = X.std()
        return self

    def transform(self, X):
        X = np.array(X)
        return (X - self.means_) / self.global_std_

    def inverse_transform(self, X):
        X = np.array(X)
        return (X * self.global_std_) + self.means_

In [ ]:
@dataclass
class SingleRun:
    model: Any
    codes: np.ndarray
    components: np.ndarray
    recon: np.ndarray
    loss: float
    hyperparameters: dict

In [ ]:
def show_closest_component_of_W_for_each_component(components, W_true, figsize=(5, 2)):
    """
    Given two arrays of numpy vectors of same shapes
    for every component in `components`, this function shows the array in `W_true`
    which has the maximum cosine similarity with the component
    """
    sims = np.abs(cosine_similarity(components, W_true))
    pairs = []
    for i in range(len(components)):
        j = np.argmax(sims[i])
        pairs.append((i, j, sims[i][j]))
    for i, j, score in pairs:
        S(
            [components[i].reshape(3, 3), W_true[j].reshape(3, 3)],
            figsize,
            mode=MODE,
            suptitle=f"similarity score={score}",
            ax_titles=["component", "ground_truth"],
            viztype="local",
        )
        plt.show()


def evaluate_recovery(W_learned, W_true, threshold=0.95):
    """
    W_learned: (n_atoms, patch_dim)
    W_true: (n_atoms, patch_dim)

    for each true atom, finds the best matching learned atom by cosine similarity
    returns fraction of true atoms recovered above threshold
    """
    W_l = W_learned / (np.linalg.norm(W_learned, axis=1, keepdims=True) + 1e-8)
    W_t = W_true / (np.linalg.norm(W_true, axis=1, keepdims=True) + 1e-8)

    sim = np.abs(W_l @ W_t.T)  # (n_atoms, n_atoms), abs because sign is arbitrary
    best_match = sim.max(
        axis=0
    )  # for each true atom, best cosine with any learned atom

    recovered = (best_match >= threshold).mean()
    print(f"Mean best cosine similarity: {best_match.mean():.4f}")
    print(f"Fraction recovered (>{threshold}): {recovered:.4f}")
    return best_match, recovered


def match_atoms(D1: np.ndarray, D2: np.ndarray):
    """
    Match atoms of D1 to atoms of D2 using the Hungarian algorithm
    on cosine distances. Assumes square dictionaries (same n_components).

    D1, D2: shape (n_components, n_features) — sklearn's components_ layout.

    Returns:
        row_ind, col_ind: matched index arrays
        matched_similarities: per-pair cosine similarities
        mean_sim: mean cosine similarity across matched pairs
    """
    # Guard against dead atoms (zero-norm rows produce NaN cosine distances)
    norms_1 = np.linalg.norm(D1, axis=1, keepdims=True)
    norms_2 = np.linalg.norm(D2, axis=1, keepdims=True)
    if np.any(norms_1 == 0) or np.any(norms_2 == 0):
        raise ValueError(
            "One or more atoms have zero norm. "
            "Remove or replace dead atoms before matching."
        )

    # cost = cosine_distances(D1, D2)          # shape (n_components, n_components), values in [0, 2]
    cost = np.abs(cosine_similarity(D1, D2))
    cost = 1 - cost

    row_ind, col_ind = linear_sum_assignment(cost)
    matched_similarities = 1 - cost[row_ind, col_ind]
    mean_sim = float(matched_similarities.mean())
    return row_ind, col_ind, matched_similarities, mean_sim


def get_live(components):
    dead = find_dead_atoms(components).numpy()
    live = np.array([i for i in range(components.shape[0]) if i not in dead])
    return components[live]


def hungarian_match(all_components: list[np.ndarray]):
    """
    Pairwise similarity matching across runs using the Hungarian algorithm.

    Args:
        all_components: list of arrays, each shape (n_components, n_features).
                        All arrays must have the same shape.

    Returns:
        upper: 1-D array of pairwise similarities for all unique pairs
        stability_score: mean of upper
        best_run_idx: index of the run most similar to all others
        pairwise_sims: (n_runs, n_runs) symmetric similarity matrix, diagonal = 1
    """
    n_runs = len(all_components)

    if n_runs < 2:
        raise ValueError("Need at least 2 runs to compute pairwise similarity.")

    shapes = [d.shape for d in all_components]
    if len(set(shapes)) != 1:
        raise ValueError(f"All dictionaries must have the same shape. Got: {shapes}")

    pairwise_sims = np.ones((n_runs, n_runs))
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            _, _, _, mean_sim = match_atoms(all_components[i], all_components[j])
            pairwise_sims[i, j] = mean_sim
            pairwise_sims[j, i] = mean_sim

    upper = pairwise_sims[np.triu_indices(n_runs, k=1)]
    stability_score = float(upper.mean())

    # Exclude self-similarity (diagonal=1) when ranking runs
    np.fill_diagonal(pairwise_sims, 0)
    mean_sim_per_run = pairwise_sims.sum(axis=1) / (n_runs - 1)
    best_run_idx = int(np.argmax(mean_sim_per_run))
    np.fill_diagonal(pairwise_sims, 1)  # restore diagonal

    return upper, stability_score, best_run_idx, pairwise_sims


def find_dead_atoms(W, threshold=0.1):
    if not isinstance(W, torch.Tensor):
        W = torch.tensor(W)

    peak = W.abs().max(dim=1).values
    peak_normalised = peak / peak.max()

    return torch.where(peak_normalised < threshold)[0]

In [ ]:
import time
import numpy as np
import torch
from pt_to_api.utils import *
from collections import defaultdict
import numpy as np
from torch import nn
from torch import optim
from pt_to_api.benchmark.core import SingleRun



class Autoencoder(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Linear(input_dim, n_components, bias=True)
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        codes = self.encoder(x)
        recon = self.decoder(codes)
        return recon, codes


def recon_loss(x, recons, sigma_eps):
    """Reconstruction loss. MSE"""
    return gauss_loss(x, recons) / (sigma_eps * sigma_eps)


def codes_loss(codes, sigma_s):
    """L2 loss on encoder"""
    return gauss_loss(codes, 0) / (sigma_s * sigma_s)


def gauss_loss(x, mean):
    loss = (x - mean) ** 2
    return loss.mean()


def weights_loss(alpha, sigma_0, W):
    """Vectorized version the Weight loss"""
    W_sq = W**2  # (C, K)
    cumsum = torch.cumsum(W_sq, dim=1)  # (C, K), cumsum[c,k] = sum W[c,0..k]^2
    phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
    phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
    comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
    comp2 = -torch.log(phi)
    return comp1.sum(dim=1).mean(), comp2.sum(dim=1).mean()


def get_scaled_hyperparamters(
    sigma_x,
    input_dim,
    n_components,
    eps_ratio=100,
    w_to_eps_ratio=5,
    alpha_constant=5000.0,
    sigma_s_rel_to_0="equal",
):
    """
    sigma_x      : std of your data
    eps_ratio    : sigma_x / sigma_eps (default 100)
    w_to_eps_ratio: sigma_0 / sigma_eps (default 5)
    alpha_constant: the c in alpha = c / sigma_0^2
    """
    # print("sigma_s_rel_to_0", sigma_s_rel_to_0)
    sigma_eps = sigma_x / eps_ratio
    # make them equal
    # if sigma_s_rel_to_0 == "equal":
    #     sigma_0 = sigma_x / np.sqrt(n_components)
    #     sigma_s = sigma_x / np.sqrt(n_components)
    # elif sigma_s_rel_to_0 == "less":
    #     sigma_s = sigma_eps * w_to_eps_ratio
    #     sigma_0 = sigma_x / (
    #         np.sqrt(n_components) * sigma_s
    #     )
    # else:
    #     sigma_0 = sigma_eps * w_to_eps_ratio
    #     sigma_s = sigma_x / (
    #         np.sqrt(n_components) * sigma_0
    #     )

    
    if sigma_s_rel_to_0 == "equal":
        # assume sigma_0 is the full columns sigma, so no need for K now
        sigma_0 = sigma_s = sigma_x
    elif sigma_s_rel_to_0 == "less":
        sigma_s = sigma_eps * w_to_eps_ratio
        sigma_0 = sigma_x / sigma_s
    else:
        sigma_0 = sigma_eps * w_to_eps_ratio
        sigma_s = sigma_x / sigma_0

    sigma_enc = sigma_s / (
        np.sqrt(input_dim) * sigma_x
    )  # from D*sigma_enc^2*sigma_x^2 = sigma_s^2
    alpha = alpha_constant / (sigma_0**2)

    return dict(
        sigma_eps=sigma_eps,
        sigma_0=sigma_0,
        sigma_s=sigma_s,
        sigma_enc=sigma_enc,
        alpha=alpha,
    )


def init_encoder_using_normal(model, sigma_enc):
    # encoder weights
    nn.init.normal_(model.encoder.weight, mean=0.0, std=sigma_enc)
    nn.init.zeros_(model.encoder.bias)


def init_model_parameters_using_normal(model, scaled_hyperparameters):
    p = scaled_hyperparameters
    sigma_0, sigma_enc = p["sigma_0"], p["sigma_enc"]

    # decoder = W, init with sigma_0
    nn.init.normal_(model.decoder.weight, mean=0.0, std=sigma_0)
    init_encoder_using_normal(model, sigma_enc)


def init_model_parameters_using_svd(model, X, n_components, scaled_hyperparameters):
    p = scaled_hyperparameters

    # svd
    _, _, Vt = np.linalg.svd(X, full_matrices=False)
    w = torch.tensor(Vt[:n_components].T, dtype=torch.float32)
    # scale
    w = (w / w.std()) * p["sigma_0"]
    model.decoder.weight.data = w

    # encoder weights
    init_encoder_using_normal(model, p["sigma_enc"])


def get_hyperparameters_and_init(
    model, X, n_components, input_dim, svd_init=False, eps_ratio=100, sigma_s_rel_to_0="equal"
):
    sigma_x = X.std()
    scaled_hyperparameters = get_scaled_hyperparamters(
        sigma_x,
        input_dim,
        n_components,
        eps_ratio=eps_ratio,
        w_to_eps_ratio=5,
        alpha_constant=5000,
        sigma_s_rel_to_0=sigma_s_rel_to_0,
    )
    if svd_init:
        init_model_parameters_using_svd(model, X, n_components, scaled_hyperparameters)
    else:
        init_model_parameters_using_normal(model, scaled_hyperparameters)
    return scaled_hyperparameters



def train(
    X,
    n_components,
    lr=1e-3,
    epochs=2000,
    batch_size=2048,
    verbose=True,
    svd_init=False,
    initialised_model=None,
    use_ln_term=True,
    sigma_s_rel_to_0="equal",
    warmup_epochs=0,
    device="mps",
) -> SingleRun:
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    """
    print("using hyperparameters")
    print("\twarmup_epochs", warmup_epochs)
    print("\tuse_ln", use_ln_term)
    print("\tsigma_s_rel_to_0", sigma_s_rel_to_0)
    print("\tsvd_init", svd_init)
    
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
    X_t = X_t.to(device)

    if initialised_model is None:
        model = Autoencoder(input_dim, n_components)
    else:
        model = initialised_model

    model = model.to(device)

    scaled_hyperparameters = get_hyperparameters_and_init(
        model, X, n_components, input_dim, svd_init, 100, sigma_s_rel_to_0
    )

    p = scaled_hyperparameters
    sigma_eps, sigma_0, sigma_s, _, alpha = (
        p["sigma_eps"],
        p["sigma_0"],
        p["sigma_s"],
        p["sigma_enc"],
        p["alpha"],
    )

    losses = defaultdict(list)

    if warmup_epochs > 0:
        warmup_with_l2(
            X_t, model, warmup_epochs, lr, batch_size, sigma_eps, sigma_s, sigma_0, verbose=verbose, device=device
        )

    optimizer = optim.Adam(model.parameters(), lr=lr)

    last_print_time = time.time()

    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples, device=device)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)

            _recon_loss = recon_loss(batch, recon, sigma_eps)
            _codes_loss = codes_loss(codes, sigma_s)
            comp1, comp2 = weights_loss(alpha, sigma_0, model.decoder.weight)
            # print(comp1.shape, comp2.shape, epoch)
            # return
            # comp1, comp2 = comp1.sum(), comp2.sum()

            # weight_loss = comp1 + comp2
            if use_ln_term:
                weight_loss = comp1 + comp2
            else:
                weight_loss = comp1

            loss = _recon_loss + weight_loss + _codes_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if verbose and epoch % 200 == 0:
            print(
                f"epoch {epoch:4d} | recon_loss {_recon_loss:.4f} weight_loss {weight_loss:.4f} codes_loss {_codes_loss:.4f} duration={time.time() - last_print_time}"
            )
            last_print_time = time.time()

    with torch.no_grad():
        recon, codes = model(X_t)

    return SingleRun(
        model.to("cpu"),
        codes.to("cpu").numpy(),
        model.decoder.weight.T.detach().to("cpu").numpy(),
        recon.to("cpu").numpy(),
        ((X_t - recon) ** 2).sum(1).mean().to("cpu"),
        scaled_hyperparameters,
    )


def warmup_with_l2(
    X_t, model, epochs, lr, batch_size, sigma_eps, sigma_s, sigma_0, verbose=True, device="mps"
):
    print("starting warmup: epochs =", epochs)
    n_samples, input_dim = X_t.shape
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples, device=device)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)
            _recon_loss = recon_loss(batch, recon, sigma_eps)
            _codes_loss = codes_loss(codes, sigma_s)
            _weights_loss = gauss_loss(model.decoder.weight, 0) / (sigma_0 * sigma_0)

            loss = _recon_loss + _codes_loss + _weights_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose and epoch % 200 == 0:
            print(f"warmup: epoch {epoch:4d} | recon_loss {_recon_loss:.4f}")



In [ ]:
def std_ratios(v1, v2):
    rat = min(v1, v2) / max(v1, v2)
    if isinstance(rat, torch.Tensor):
        return rat.item()
    return rat

def get_metrics_from_run(run: SingleRun, W_true):
    _, _, sim_vector, mean_sim = match_atoms(W_true, run.components)
    mx = run.model.decoder.weight.max()
    thres = mx / 1000
    
    decoder_maxes = run.model.decoder.weight.max(dim=0)[0]
    return {
        "mse": run.loss,
        "gram": gram_orthogonality_error(run.components.T),
        "decoder_maxes_mean_ratio": std_ratios(
            decoder_maxes.mean().item(), run.hyperparameters["sigma_0"]
        ),
        "encoder_sigma_ratio": std_ratios(
            run.model.encoder.weight.std().item(), run.hyperparameters["sigma_enc"]
        ),
        "decoder_sigma_ratio": std_ratios(
            run.model.decoder.weight.std().item(), run.hyperparameters["sigma_0"]
        ),
        "mean_sim": mean_sim,
        "vec_sim": sim_vector,
    }

def print_summary(X, W_true, codes_true, run: SingleRun):
    print("standard devications")
    print("\tX:", X.std())
    print("\tW_true:", W_true.std())
    print("\tcodes_true:", codes_true.std())
    print("\tcodes:", run.codes.std(), "sigma_s", run.hyperparameters["sigma_s"])
    print("\tcomponents:", run.components.std())
    print("\trecon:", run.recon.std())
    print("\tdecoder:", run.model.decoder.weight.std().item(), "sigma_0:", run.hyperparameters["sigma_0"])
    print("\tencoder:", run.model.encoder.weight.std().item(), "sigma_enc:", run.hyperparameters["sigma_enc"])
    print("\tMSE:", (X - run.recon).std(), "sigma_eps:", run.hyperparameters["sigma_eps"])

# Start experiment

In [ ]:
def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]

def generate_synthetic_patches(
    patch_dim=72,
    n_components=10,
    k=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
):
    rng = np.random.RandomState(seed)

    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses at most k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        k_i = rng.randint(1, k + 1)  # active atoms: 1..k
        idx = rng.choice(n_components, k_i, replace=False)
        codes_true[i, idx] = rng.randn(k_i)

    X = codes_true @ W_true

    scale = sigma_x / X.std()
    X *= scale
    W_true *= scale  # keeps codes_true @ W_true ≈ X
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition

In [ ]:
gen_fn = generate_synthetic_patches

In [ ]:
def aggregate_metrics(dicts):
    keys = dicts[0].keys()
    result = {}
    for key in keys:
        vals = [d[key] for d in dicts]
        if key == 'vec_sim':
            result[key] = np.stack(vals).mean(axis=0)
        elif key == 'mse':
            result[key] = sum(v.item() for v in vals) / len(vals)
        else:
            result[key] = sum(vals) / len(vals)
    return result

NUM_RUNS_PER_TEST = 3

# Relation between $\sigma_s$ and $\sigma_0$

We can either set $\sigma_s$ to $c\sigma_{\epsilon}$ and derive $\sigma_0$ from it. or the other way round.  
This will determine which one is bigger.  

The relationship we use is simply $R\sigma_0^2\sigma_s^2 = \sigma_x^2$ ($R$ is the number of atoms, or rows in components).  

- We can either set one of them relative to $sigma_eps$ and derive the other
- We can make them equal, ie. $\sigma_0 = \sigma_s = \frac{\sigma_x}{\sqrt{R}}$

Below we will empirically see that setting them to equal is a good condition, for smaller examples. The test will continue for bigger examples later.  


We test for dimensions = `[10, 100]` with `[20%, 50%, 80%]` active atoms.   

I need simple things to test on first.   


Things that change:
- number of components
- number of dims
- number of components active at a time

The simplest is checking how many atoms we can recover, we keep all the components active to keep it simple. For a given dimension.  
We will assume that data = 100 * components-number.  

In [ ]:
import math

dims_list = [10, 100]
atoms_ratio = [0.2, 0.5]
samples_to_perturb_ratio_set = [0.1, 0.2, 0.3, 0.5]
atoms_to_perturb_ratio_set = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]


sigma_s_rel_to_0_set = ["equal", "less", "greater"]

term3_set = sigma_s_rel_to_0_set
kwarg_key = "sigma_s_rel_to_0"


metrics = {}

for dim in dims_list:
    for a in atoms_ratio:
        for atoms_to_perturb_ratio in atoms_to_perturb_ratio_set:
            for samples_to_perturb_ratio in samples_to_perturb_ratio_set:
                for term3 in term3_set:
                
                    print("#######", dim, a, atoms_to_perturb_ratio, samples_to_perturb_ratio)
                    gc.collect()
                    atoms = math.ceil(dim*a)
                    k = atoms
                    atoms_to_perturb = math.ceil(atoms * atoms_to_perturb_ratio)
                    n_samples = 100*atoms
                    
                    
                    X, W_true, codes_true, dim_partition, _ = generate_corrupted_dataset(
                        n_atoms_to_perturb=atoms_to_perturb,
                        ratio=samples_to_perturb_ratio,
                        per_sample=True,
                        patch_dim=dim,
                        n_components=atoms,
                        k=k,
                        n_samples=n_samples,
                        noise_std=0.01,
                        seed=42,
                        sigma_x=1,
                    )
            
                    scaler = MeanPerDimGlobalStdScaler().fit(X)
                    X_scaled = scaler.transform(X)
    
                    mets = []
                    for run_idx in range(NUM_RUNS_PER_TEST):
                        print("RUN:", run_idx)
                        kwargs = {kwarg_key: term3}
                        run = train(X_scaled, atoms, 1e-3, epochs=4000, **kwargs)
                        mets.append(get_metrics_from_run(run, W_true))
                        gc.collect()
                    
                    metrics[(dim, a, atoms_to_perturb, samples_to_perturb_ratio, term3)] = aggregate_metrics(mets)

## Results

If you look at the table, you'll see that the sigma_s_to_0 gives mean simimilary (the column `mean_sim`) = 99%.  

Surprisingly, we see that if we set $\sigma_s < \sigma_0$, `dims=10` gives 55% mean similarity. The reason is the corresponding MSE. Its much higher than the equal case, we are not able to reconstruct well, so every atom is just a disjoint noise.  

For `dims=100`, $\sigma_s > \sigma_0$, we see 30-60% similarity. The MSE is fine too. On closer inspection, you'll see that the disjoint loss has driven all atoms to near 0. If you look at individual components of a single reconstruction, you'll see random atoms lighting up. This is because the codes have too much variance and are basically taking care of getting the reconstruction.  

It's hard to say why this happens. It's best to ignore that. For now, $\sigma_s = \sigma_0$ gives the best results.  
It also has the good property to simply remove the relation with $\sigma_eps$, which we can use to freely guide reconstruction.  

The next experiments will use the "equal" method

In [ ]:
mets = []
for k, v in metrics.items():
    m = v.copy()
    m["dims"] = k[0]
    m["atom_ratio"] = k[1]
    m["atoms_to_perturb"] = k[2]
    m["samples_to_perturb_ratio"] = k[3]
    m["term3"] = k[4]
    mets.append(m)

df = pd.DataFrame(mets)
df = df.drop(columns=["vec_sim"])

df

In [ ]:
# all equal values
df[df["term3"] == "equal"]

In [ ]:
# all less
df[df["term3"] == "less"]

In [ ]:
# all greater
df[df["term3"] == "greater"]

# log term usefulness

Closely inspecting the gradients generally tells us that the log term has very small gradients through out.  
The log term is basically pushing in the opposite direction from disjointness. It wants to keep weights "non-zero".  
The signal is very low though compared to the actual disjoint term. And we anyways rely on correct reconstruction to give us non-zero weights.  

This section checks if descent becomes easier in the synthetic case if we give up the log term. It is useful to only check the results in the previous section with the `sigma_s_to_0 = equal` case.  


In [ ]:
import math

dims_list = [10, 100]
atoms_ratio = [0.2, 0.5]
samples_to_perturb_ratio_set = [0.1, 0.2, 0.3, 0.5]
atoms_to_perturb_ratio_set = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]



term3_set = [True, False]
kwarg_key = "use_ln_term"



metrics = {}

for dim in dims_list:
    for a in atoms_ratio:
        for atoms_to_perturb_ratio in atoms_to_perturb_ratio_set:
            for samples_to_perturb_ratio in samples_to_perturb_ratio_set:
                for term3 in term3_set:
                
                    print("#######", dim, a, atoms_to_perturb_ratio, samples_to_perturb_ratio)
                    gc.collect()
                    atoms = math.ceil(dim*a)
                    k = atoms
                    atoms_to_perturb = math.ceil(atoms * atoms_to_perturb_ratio)
                    n_samples = 100*atoms
                    
                    
                    X, W_true, codes_true, dim_partition, _ = generate_corrupted_dataset(
                        n_atoms_to_perturb=atoms_to_perturb,
                        ratio=samples_to_perturb_ratio,
                        per_sample=True,
                        patch_dim=dim,
                        n_components=atoms,
                        k=k,
                        n_samples=n_samples,
                        noise_std=0.01,
                        seed=42,
                        sigma_x=1,
                    )
            
                    scaler = MeanPerDimGlobalStdScaler().fit(X)
                    X_scaled = scaler.transform(X)
    
                    mets = []
                    for run_idx in range(NUM_RUNS_PER_TEST):
                        print("RUN:", run_idx)
                        kwargs = {kwarg_key: term3}
                        run = train(X_scaled, atoms, 1e-3, epochs=4000, **kwargs)
                        mets.append(get_metrics_from_run(run, W_true))
                        gc.collect()
                    
                    metrics[(dim, a, atoms_to_perturb, samples_to_perturb_ratio, term3)] = aggregate_metrics(mets)


In [ ]:
mets = []
for k, v in metrics.items():
    m = v.copy()
    m["dims"] = k[0]
    m["atom_ratio"] = k[1]
    m["atoms_to_perturb"] = k[2]
    m["samples_to_perturb_ratio"] = k[3]
    m["term3"] = k[4]
    mets.append(m)

df = pd.DataFrame(mets)
df = df.drop(columns=["vec_sim"])

df

## Results

We don't see a lot of difference. Although removing the `ln` component did bring down `mean_sim` in one case (row 3).  
It has also degraded the MSE in some cases, so this is not very conclusive.  

We will need to test this on other datasets to see if there is problem with the component. Otherwise we go ahead while staying faithful to the probabilistic model.  

# Train for reconstruction first, then other penalties

We add a warmup where we simply train for reconstruction loss, with basic L2 penalties on W and S to maintain their weight distribution stds. 
This will add warmup epochs to the training process

In [ ]:
import math

dims_list = [10, 100]
atoms_ratio = [0.2, 0.5]
samples_to_perturb_ratio_set = [0.1, 0.2, 0.3, 0.5]
atoms_to_perturb_ratio_set = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]



term3_set = [0, 800]
kwarg_key = "warmup_epochs"

metrics = {}

for dim in dims_list:
    for a in atoms_ratio:
        for atoms_to_perturb_ratio in atoms_to_perturb_ratio_set:
            for samples_to_perturb_ratio in samples_to_perturb_ratio_set:
                for term3 in term3_set:
                
                    print("#######", dim, a, atoms_to_perturb_ratio, samples_to_perturb_ratio)
                    gc.collect()
                    atoms = math.ceil(dim*a)
                    k = atoms
                    atoms_to_perturb = math.ceil(atoms * atoms_to_perturb_ratio)
                    n_samples = 100*atoms
                    
                    
                    X, W_true, codes_true, dim_partition, _ = generate_corrupted_dataset(
                        n_atoms_to_perturb=atoms_to_perturb,
                        ratio=samples_to_perturb_ratio,
                        per_sample=True,
                        patch_dim=dim,
                        n_components=atoms,
                        k=k,
                        n_samples=n_samples,
                        noise_std=0.01,
                        seed=42,
                        sigma_x=1,
                    )
            
                    scaler = MeanPerDimGlobalStdScaler().fit(X)
                    X_scaled = scaler.transform(X)
    
                    mets = []
                    for run_idx in range(NUM_RUNS_PER_TEST):
                        print("RUN:", run_idx)
                        kwargs = {kwarg_key: term3}
                        run = train(X_scaled, atoms, 1e-3, epochs=4000, **kwargs)
                        mets.append(get_metrics_from_run(run, W_true))
                        gc.collect()
                    
                    metrics[(dim, a, atoms_to_perturb, samples_to_perturb_ratio, term3)] = aggregate_metrics(mets)


In [ ]:
mets = []
for k, v in metrics.items():
    m = v.copy()
    m["dims"] = k[0]
    m["atom_ratio"] = k[1]
    m["atoms_to_perturb"] = k[2]
    m["samples_to_perturb_ratio"] = k[3]
    m["term3"] = k[4]
    mets.append(m)

df = pd.DataFrame(mets)
df = df.drop(columns=["vec_sim"])

df

## Results

not much difference in the pure case

# SVD initialisation

SVD initialisation gives very deterministic outputs. It also gives quite good outputs infact. It is useful to add a comparison.  


In [ ]:
import math

dims_list = [10, 100]
atoms_ratio = [0.2, 0.5]
samples_to_perturb_ratio_set = [0.1, 0.2, 0.3, 0.5]
atoms_to_perturb_ratio_set = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]



term3_set = [True, False]
kwarg_key = "svd_init"


metrics = {}

for dim in dims_list:
    for a in atoms_ratio:
        for atoms_to_perturb_ratio in atoms_to_perturb_ratio_set:
            for samples_to_perturb_ratio in samples_to_perturb_ratio_set:
                for term3 in term3_set:
                
                    print("#######", dim, a, atoms_to_perturb_ratio, samples_to_perturb_ratio)
                    gc.collect()
                    atoms = math.ceil(dim*a)
                    k = atoms
                    atoms_to_perturb = math.ceil(atoms * atoms_to_perturb_ratio)
                    n_samples = 100*atoms
                    
                    
                    X, W_true, codes_true, dim_partition, _ = generate_corrupted_dataset(
                        n_atoms_to_perturb=atoms_to_perturb,
                        ratio=samples_to_perturb_ratio,
                        per_sample=True,
                        patch_dim=dim,
                        n_components=atoms,
                        k=k,
                        n_samples=n_samples,
                        noise_std=0.01,
                        seed=42,
                        sigma_x=1,
                    )
            
                    scaler = MeanPerDimGlobalStdScaler().fit(X)
                    X_scaled = scaler.transform(X)
    
                    mets = []
                    for run_idx in range(NUM_RUNS_PER_TEST):
                        print("RUN:", run_idx)
                        kwargs = {kwarg_key: term3}
                        run = train(X_scaled, atoms, 1e-3, epochs=4000, **kwargs)
                        mets.append(get_metrics_from_run(run, W_true))
                        gc.collect()
                    
                    metrics[(dim, a, atoms_to_perturb, samples_to_perturb_ratio, term3)] = aggregate_metrics(mets)


In [ ]:
mets = []
for k, v in metrics.items():
    m = v.copy()
    m["dims"] = k[0]
    m["atom_ratio"] = k[1]
    m["atoms_to_perturb"] = k[2]
    m["samples_to_perturb_ratio"] = k[3]
    m["term3"] = k[4]
    mets.append(m)

df = pd.DataFrame(mets)
df = df.drop(columns=["vec_sim"])

df

## Results